# 예제 05. RNN과 LSTM 비교 · 입력 구간 실험
빅데이터프로그래밍 · 11주차

## 목표
- 같은 데이터에서 RNN과 LSTM을 비교한다
- 사인파 주기와 입력 구간 길이를 바꿔 실험한다
- 결과표와 그래프를 만든다

과제와 같은 형식입니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

T = 1000
loss_fn = nn.MSELoss()
EPOCHS = 30


## 1. 공통 함수


In [ ]:
def make_series(freq=0.05, noise=0.05, T=T, seed=42):
    np.random.seed(seed)
    t = np.arange(T)
    return np.sin(t * freq) + np.random.randn(T) * noise


class SeriesDataset(Dataset):
    def __init__(self, arr, seq_len):
        self.arr = torch.tensor(arr, dtype=torch.float32)
        self.seq_len = seq_len
    def __len__(self):
        return len(self.arr) - self.seq_len
    def __getitem__(self, i):
        return (self.arr[i:i+self.seq_len].unsqueeze(-1),
                self.arr[i+self.seq_len].unsqueeze(-1))


class Forecast(nn.Module):
    def __init__(self, kind="lstm", hidden=64):
        super().__init__()
        layer = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
        self.rnn = layer(1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])


def make_loaders(series, seq_len, batch=32):
    n = int(len(series) * 0.8)
    tr = DataLoader(SeriesDataset(series[:n], seq_len), batch_size=batch, shuffle=True)
    va = DataLoader(SeriesDataset(series[n:], seq_len), batch_size=batch, shuffle=False)
    return tr, va


def run(kind, series, seq_len, epochs=EPOCHS, lr=1e-3, seed=42):
    torch.manual_seed(seed)
    tr, va = make_loaders(series, seq_len)
    model = Forecast(kind).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    start = time.time()
    for _ in range(epochs):
        model.train()
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in va:
            preds.append(model(xb.to(device)).cpu()); trues.append(yb)
    p = torch.cat(preds).squeeze().numpy(); tt = torch.cat(trues).squeeze().numpy()
    return {"model": model, "pred": p, "true": tt,
            "mse": float(((p-tt)**2).mean()),
            "mae": float(np.abs(p-tt).mean()),
            "rmse": float(np.sqrt(((p-tt)**2).mean())),
            "time": time.time()-start,
            "params": sum(q.numel() for q in model.parameters())}


## 2. RNN vs LSTM vs GRU


In [ ]:
series = make_series()
SEQ = 40

results = {}
for kind in ["rnn", "lstm", "gru"]:
    results[kind.upper()] = run(kind, series, SEQ)
    r = results[kind.upper()]
    print(f"{kind.upper():5s} RMSE {r['rmse']:.5f}  {r['time']:.1f}초  파라미터 {r['params']:,}개")


In [ ]:
pd.DataFrame([{"모델": k, "파라미터": f"{r['params']:,}",
               "MAE": round(r["mae"], 5), "RMSE": round(r["rmse"], 5),
               "학습 시간(초)": round(r["time"], 1)}
              for k, r in results.items()])


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
for ax, (name, r) in zip(axes, results.items()):
    ax.plot(r["true"], label="실제", linewidth=1.4)
    ax.plot(r["pred"], label="예측", linewidth=1.2, linestyle="--")
    ax.set_title(f"{name}  RMSE {r['rmse']:.4f}"); ax.grid(alpha=.3)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 3. 입력 구간 길이 실험


In [ ]:
rows = []
for seq in [5, 10, 20, 40, 80]:
    r = run("lstm", series, seq)
    rows.append({"seq_len": seq, "표본 수": len(SeriesDataset(series[:800], seq)),
                 "RMSE": round(r["rmse"], 5), "시간(초)": round(r["time"], 1)})
seq_df = pd.DataFrame(rows)
print(seq_df.to_string(index=False))

plt.figure(figsize=(7, 3.4))
plt.plot(seq_df["seq_len"], seq_df["RMSE"], marker="o")
plt.xlabel("입력 구간 길이"); plt.ylabel("RMSE"); plt.grid(alpha=.3)
plt.title("구간 길이와 예측 오차")
plt.show()


## 4. 사인파 주기 실험
주기가 짧으면(빠르게 흔들리면) 같은 구간에 더 많은 변화가 담깁니다.


In [ ]:
rows = []
for freq in [0.02, 0.05, 0.1, 0.3]:
    s = make_series(freq=freq)
    period = 2*np.pi/freq
    r = run("lstm", s, SEQ)
    rows.append({"freq": freq, "주기(시점)": round(period, 1),
                 "구간에 담긴 주기 수": round(SEQ/period, 2),
                 "RMSE": round(r["rmse"], 5)})
pd.DataFrame(rows)


## 5. 잡음 수준 실험


In [ ]:
rows = []
for noise in [0.0, 0.05, 0.15, 0.3]:
    s = make_series(noise=noise)
    r = run("lstm", s, SEQ)
    rows.append({"잡음 크기": noise, "RMSE": round(r["rmse"], 5)})
pd.DataFrame(rows)


잡음이 커지면 RMSE도 커집니다. 잡음 자체는 예측할 수 없기 때문입니다 — 모델의 잘못이 아닙니다.


## 6. RNN과 LSTM을 긴 구간에서 비교


In [ ]:
rows = []
for seq in [10, 40, 100]:
    for kind in ["rnn", "lstm"]:
        r = run(kind, series, seq, epochs=20)
        rows.append({"seq_len": seq, "모델": kind.upper(), "RMSE": round(r["rmse"], 5)})
long_df = pd.DataFrame(rows).pivot(index="seq_len", columns="모델", values="RMSE")
print(long_df.to_string())


## 직접 해보기
1. 두 주기를 섞은 시계열에서도 같은 결론이 나오나요?
2. `hidden` 을 16과 128로 바꿔 비교하세요.
3. 층을 2개로 쌓으면 (`num_layers=2`) 결과가 좋아지나요?


In [ ]:
# 여기에 작성하세요
